# S2: challenge-style samples (Charan et al., arXiv:2209.07519)
Run after S1. Run All. Outputs go to the `S32` folder.

In [2]:
# ==== S2: BUILD SAMPLES THE WAY THE DEEPSENSE 2022 CHALLENGE DOES (Charan et al., arXiv:2209.07519)
# run in the ODT folder, after S1. needs: numpy, pandas (pyproj optional: exact UTM)
"""
Challenge sample = 5 consecutive base-station observations (image, LiDAR, radar at t-4 ... t) + the user's GPS
at the first two of them (t-4, t-3); label = optimal beam (0..63) at t.
Position processing as in the paper:  lat/lon -> UTM x, y  ->  UE minus BS  ->  min-max normalisation per column.
Our changes (stricter): the 5 frames are one drive, contiguous, no repeat / gap > 0.3 s / bad power;
split by whole drives from S1; min / max taken from the training samples only.
"""
import os, re, sys, json, argparse
import numpy as np
import pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument("--scen", default="32")
ap.add_argument("--r", type=int, default=5, help="base-station observations per sample")
ap.add_argument("--r_gps", type=int, default=2, help="GPS observations (the first ones of the sample)")
ap.add_argument("--max_dt", type=float, default=0.3)
A = ap.parse_args([])   # notebook: settings are the defaults above
D = f"S{A.scen}"; SD = f"scenario{A.scen}"


class Tee:
    def __init__(self, path):
        self.f, self.o = open(path, "w", encoding="utf-8"), sys.stdout

    def write(self, s):
        self.o.write(s); self.f.write(s); self.f.flush()

    def flush(self):
        self.o.flush(); self.f.flush()


while hasattr(sys.stdout, "o"):
    sys.stdout = sys.stdout.o
sys.stdout = Tee(os.path.join(D, "S2_report.txt"))
df = pd.read_csv(os.path.join(D, f"S{A.scen}_frames.csv"))
P = np.load(os.path.join(D, f"S{A.scen}_power.npy"))
raw = pd.read_csv(next(os.path.join(SD, f) for f in sorted(os.listdir(SD)) if f.endswith(".csv") and "dev" in f.lower())
                  if any("dev" in f.lower() for f in os.listdir(SD)) else os.path.join(SD, sorted(f for f in os.listdir(SD) if f.endswith(".csv"))[0]))
assert len(raw) == len(df), "S1 table and CSV differ: re-run S1"

# ------------------------------------------------------------------ 1. GPS -> UTM x, y (metres), UE minus BS
def latlon(p):
    p = str(p).strip(); p = os.path.join(SD, p[2:] if p.startswith("./") else p)
    try:
        v = [float(x) for x in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", open(p).read())][:2]
        return v if len(v) == 2 else [np.nan, np.nan]
    except OSError:
        return [np.nan, np.nan]


bs_col = next(c for c in raw.columns if c.lower() == "unit1_loc")
ue_col = next(c for c in raw.columns if c.lower().startswith("unit2_loc"))
BS = np.array([latlon(p) for p in raw[bs_col]]); UE = np.array([latlon(p) for p in raw[ue_col]])
try:
    from pyproj import Transformer
    zone = int((np.nanmedian(BS[:, 1]) + 180) // 6) + 1
    tr = Transformer.from_crs("EPSG:4326", f"EPSG:{32600 + zone if np.nanmedian(BS[:, 0]) >= 0 else 32700 + zone}", always_xy=True)
    bx, by = tr.transform(BS[:, 1], BS[:, 0]); ux, uy = tr.transform(UE[:, 1], UE[:, 0]); how = f"UTM zone {zone} (pyproj)"
    XY = np.c_[np.asarray(ux) - np.asarray(bx), np.asarray(uy) - np.asarray(by)]
except ImportError:
    R = 6371000.0; how = "local flat-earth projection (pyproj not installed)"
    XY = np.c_[np.radians(UE[:, 1] - BS[:, 1]) * np.cos(np.radians(BS[:, 0])) * R, np.radians(UE[:, 0] - BS[:, 0]) * R]
print(f"[1] GPS -> x, y with {how}; UE minus BS. x range {np.nanmin(XY[:, 0]):.1f}..{np.nanmax(XY[:, 0]):.1f} m, "
      f"y range {np.nanmin(XY[:, 1]):.1f}..{np.nanmax(XY[:, 1]):.1f} m")

# ------------------------------------------------------------------ 2. samples of r contiguous frames inside one drive
seq = df["seq"].to_numpy(); dt = df["dt"].to_numpy(); usable = df["usable"].to_numpy()
rows = []
for t in range(A.r - 1, len(df)):
    w = np.arange(t - A.r + 1, t + 1)
    if (seq[w] == seq[t]).all() and usable[w].all() and np.all(dt[w[1:]] <= A.max_dt) and np.isfinite(XY[w[:A.r_gps]]).all():
        rows.append(w)
rows = np.array(rows)
assert len(rows), f"no window of {A.r} contiguous usable frames (check dt in S1: median spacing must be under --max_dt {A.max_dt} s)"
split = df["split"].to_numpy()[rows[:, -1]]
print(f"\n[2] samples of {A.r} frames (GPS at the first {A.r_gps}): {len(rows)} "
      f"(from {int(usable.sum())} usable frames; lost at drive starts, gaps and dropped frames)")
for nm in ("train", "val", "test"):
    print(f"    {nm:5s} {np.sum(split == nm):5d} samples")

# ------------------------------------------------------------------ 3. min-max normalisation, fitted on training samples
pos_m = XY[rows[:, :A.r_gps]]                                                  # (S, r_gps, 2) metres
tr_m = pos_m[split == "train"].reshape(-1, 2)
mn, mx = tr_m.min(0), tr_m.max(0)
pos = (pos_m - mn) / np.maximum(mx - mn, 1e-9)
outside = np.mean((pos[split != "train"] < 0) | (pos[split != "train"] > 1))
print(f"\n[3] min-max from training: x {mn[0]:.1f}..{mx[0]:.1f} m, y {mn[1]:.1f}..{mx[1]:.1f} m | "
      f"val/test values outside [0, 1]: {100 * outside:.2f} %")

# ------------------------------------------------------------------ 4. label = beam at t (the r-th frame)
beam = df["beam"].to_numpy()[rows[:, -1]]
for nm in ("train", "val", "test"):
    b = beam[split == nm]
    print(f"    {nm:5s} beams {b.min()}..{b.max()} | most common beam {np.bincount(b, minlength=64).argmax()} "
          f"({100 * np.bincount(b, minlength=64).max() / len(b):.1f} %)")
same_as_first = np.mean(df["beam"].to_numpy()[rows[:, 0]] == beam)
print(f"    beam at t equals beam at t-{A.r - 1}: {100 * same_as_first:.1f} % (how much the beam moves within a sample)")

np.savez_compressed(os.path.join(D, "S2_samples.npz"), row=rows, seq=seq[rows[:, -1]], split=split, beam=beam,
                    power=P[rows[:, -1]], pos=pos.astype(np.float32), pos_m=pos_m.astype(np.float32), minmax=np.r_[mn, mx],
                    img=df["img_path"].to_numpy()[rows].astype(str), lidar=df["lidar_path"].to_numpy()[rows].astype(str),
                    radar=df["radar_path"].to_numpy()[rows].astype(str))
print(f"\nsaved {D}/S2_samples.npz")

while hasattr(sys.stdout, "o"):        # back to normal printing
    sys.stdout = sys.stdout.o